# RETECO SemEval-2027 – Phase 1: Setup & BM25 Baseline Reproduction

**Sub-track:** 1a (Temporal Retrieval)  
**Data version:** v1.1 (deduplicated)  
**Splits evaluated here:** train + dev (gold labels present for both)  
**Run this notebook top-to-bottom from a clean state.**

In [ ]:
# ── Environment & versions ────────────────────────────────────────────────────
import sys, platform
import importlib
from pathlib import Path

SEED = 42
DATA_ROOT = Path('../data/raw')
OUT_ROOT  = Path('../outputs')
OUT_ROOT.mkdir(exist_ok=True)

# Add src/ to path
sys.path.insert(0, str(Path('../src').resolve()))

print(f'Python : {sys.version}')
print(f'OS     : {platform.platform()}')
for pkg in ['numpy','pandas','sklearn','rank_bm25','pytrec_eval','tqdm']:
    try:
        m = importlib.import_module(pkg)
        print(f'  {pkg}: {getattr(m,"__version__","ok")}')
    except ImportError:
        print(f'  {pkg}: NOT INSTALLED')

In [ ]:
# ── 1. Inspect raw file structure ─────────────────────────────────────────────
import os

track1_root = DATA_ROOT / 'track1_tempo'
domains_found = sorted([d.name for d in track1_root.iterdir() if d.is_dir()])
print(f'Track-1 domains found ({len(domains_found)}): {domains_found}')

# Show file layout for first domain
sample_domain = domains_found[0]
print(f'\nFiles in {sample_domain}:')
for f in sorted((track1_root / sample_domain).iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:40s} {size_kb:8.1f} KB')

In [ ]:
# ── 2. Print schema examples (first record of each file type) ─────────────────
import json

d_dir = track1_root / sample_domain

def first_line(path):
    with open(path, encoding='utf-8') as f:
        for line in f:
            if line.strip():
                return json.loads(line)
    return None

print('=== documents.jsonl (first record) ===')
doc0 = first_line(d_dir / 'documents.jsonl')
print(f'  fields: {list(doc0.keys())}')
print(f'  id[:50]: {doc0["id"][:50]}')
print(f'  content[:100]: {doc0["content"][:100]}')

print('\n=== examples_train.jsonl (first record) ===')
q0 = first_line(d_dir / 'examples_train.jsonl')
print(f'  fields: {list(q0.keys())}')
print(f'  id: {q0["id"]}')
print(f'  query[:100]: {q0["query"][:100]}')
print(f'  gold_ids: {q0["gold_ids"]}')
print(f'  gold_answers[:1]: {q0["gold_answers"][:1]}')

print('\n=== qrels_train.txt (first 3 lines) ===')
with open(d_dir / 'qrels_train.txt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 3: break
        print(' ', line.rstrip())

In [ ]:
# ── 3. Per-domain corpus and query counts (all domains) ───────────────────────
import pandas as pd

rows = []
for domain in domains_found:
    d = track1_root / domain
    n_docs = sum(1 for l in open(d/'documents.jsonl', encoding='utf-8') if l.strip())
    n_train = sum(1 for l in open(d/'examples_train.jsonl', encoding='utf-8') if l.strip())
    n_dev   = sum(1 for l in open(d/'examples_dev.jsonl', encoding='utf-8') if l.strip())
    rows.append({'domain': domain, 'n_docs': n_docs, 'n_train_q': n_train, 'n_dev_q': n_dev})

df_counts = pd.DataFrame(rows).sort_values('n_docs')
print(df_counts.to_string(index=False))
print(f'\nTotal docs: {df_counts.n_docs.sum():,}')
print(f'Total train queries: {df_counts.n_train_q.sum():,}')
print(f'Total dev queries: {df_counts.n_dev_q.sum():,}')

df_counts.to_csv(OUT_ROOT / 'tables' / 'domain_counts.csv', index=False)

In [ ]:
# ── 4. Choose 3 domains (small-to-moderate corpus) ───────────────────────────
# Selection criterion: bottom-3 by n_docs so BM25 runs fast on CPU,
# but ensuring each has >=50 dev queries for reliable nDCG@10 estimates.

# Show candidates
print('Smallest-corpus candidates:')
print(df_counts[df_counts.n_dev_q >= 30][['domain','n_docs','n_train_q','n_dev_q']].head(6).to_string(index=False))

In [ ]:
# ── 5. BM25 baseline reproduction (our scorer vs official) ────────────────────
import sys, time
sys.path.insert(0, '../src')
from data_utils import load_corpus, load_queries_1a, load_qrels
from bm25_retriever import BM25Retriever
from evaluate import evaluate_run, pytrec_evaluate, write_trec_run
from pathlib import Path

# We will run on ALL 13 domains, then macro-average, to reproduce the 
# official 0.1147 number for dev (v1.1 data)

track1_root = Path('../data/raw/track1_tempo')
domains_found = sorted([d.name for d in track1_root.iterdir() if d.is_dir()])

results = {}  # domain -> {split: ndcg}
run_dir = Path('../outputs/runs')
run_dir.mkdir(exist_ok=True)

for domain in domains_found:
    d_dir = track1_root / domain
    print(f'\n── {domain} ──')
    t0 = time.time()
    doc_ids, doc_texts = load_corpus(d_dir)
    bm25 = BM25Retriever(doc_ids, doc_texts)
    print(f'  indexed {len(doc_ids):,} docs in {time.time()-t0:.1f}s')
    
    results[domain] = {}
    for split in ('train', 'dev'):
        queries = [(q['id'], q['query']) for q in load_queries_1a(d_dir, split)]
        qrels   = load_qrels(d_dir, split)
        run     = bm25.batch_search(queries, top_k=100)
        # convert to list-of-tuples format for write_trec_run
        run_for_write = {qid: ranked for qid, ranked in run.items()}
        write_trec_run(run_dir / f'bm25_{domain}_{split}.trec', run_for_write)
        # evaluate
        run_lists = {qid: [d for d,_ in ranked] for qid, ranked in run.items()}
        res = evaluate_run(run_lists, qrels)
        res_pt = pytrec_evaluate(run_lists, qrels)
        results[domain][split] = res['ndcg@10']
        print(f'  {split:5s} nDCG@10: {res["ndcg@10"]:.4f}  (pytrec: {res_pt["ndcg@10"]:.4f})  [{res["num_topics"]} queries]')

print('\n=== MACRO-AVERAGE OVER ALL 13 DOMAINS ===')
for split in ('train','dev'):
    macro = sum(results[d][split] for d in domains_found) / len(domains_found)
    print(f'  {split:5s}: {macro:.4f}  (reference: train=0.1075, dev=0.1147)')

In [ ]:
# ── 6. Per-domain results table + domain selection ────────────────────────────
import pandas as pd
import json

rows = []
for domain in domains_found:
    rows.append({
        'domain': domain,
        'n_docs': df_counts[df_counts.domain==domain].n_docs.values[0],
        'n_dev_q': df_counts[df_counts.domain==domain].n_dev_q.values[0],
        'bm25_train': results[domain]['train'],
        'bm25_dev': results[domain]['dev'],
    })

df_res = pd.DataFrame(rows).sort_values('n_docs')
print(df_res.to_string(index=False))

# Save
df_res.to_csv('../outputs/tables/bm25_all_domains.csv', index=False)
json.dump(results, open('../outputs/runs/bm25_results.json','w'), indent=2)

# ── DOMAIN SELECTION ──
# Choose 3 smallest-corpus domains with >=30 dev queries
candidates = df_res[df_res.n_dev_q >= 30].nsmallest(3, 'n_docs')
CHOSEN_DOMAINS = candidates.domain.tolist()
print(f'\n>>> CHOSEN DOMAINS: {CHOSEN_DOMAINS}')
print('Justification: smallest n_docs among domains with >=30 dev queries,')
print('enabling fast BM25/TF-IDF/dense indexing on CPU within time budget.')
print(candidates[['domain','n_docs','n_dev_q']].to_string(index=False))

# Persist choice
json.dump({'chosen_domains': CHOSEN_DOMAINS, 'criterion': 'smallest n_docs, >=30 dev queries'},
          open('../outputs/runs/chosen_domains.json','w'), indent=2)

In [ ]:
# ── 7. Reproduce official macro-average (cross-check) ─────────────────────────
print('=== CHECKPOINT 1 SUMMARY ===')
print(f'Data version: v1.1 (deduplicated, 25 Sep 2026)')
print(f'Track-1 domains: {domains_found}')
print(f'Total Track-1 corpus docs: {df_counts.n_docs.sum():,}')
print()
for split in ('train','dev'):
    macro = sum(results[d][split] for d in domains_found) / len(domains_found)
    ref = {'train': 0.1075, 'dev': 0.1147}[split]
    diff = abs(macro - ref)
    status = 'PASS' if diff < 0.005 else 'MISMATCH'
    print(f'  BM25 {split:5s}: {macro:.4f}  ref={ref}  diff={diff:.4f}  [{status}]')
print()
print(f'Chosen domains for phases 2-6: {CHOSEN_DOMAINS}')
print(f'  (smallest corpus, >=30 dev queries each)')
print()
print('NOTE: No Java required. Using starter-kit pure-Python BM25 (k1=0.9, b=0.4).')
print('pytrec_eval used for cross-check (matches our implementation).')